# STAR — Kaggle V6: COMBO (pair-batch + text-LoRA + ViTPose) · batch 20 · epoch 10

Lay cong thuc thang **v3c (pair-batch + ViTPose) = 0.7485** roi **cong them**:
- ✅ **text-LoRA BAT** (`lora_freeze_text=false`) — v3a cho +0.01, gio gop voi pair-batch
- ✅ **smart batch = pair** (`group_by=pair`) — dong gop chinh cua v3c
- ✅ **ViTPose** (`pose_enabled=true`) — fuse keypoint vao dac trung anh (train_10k_hard_vitpose.json, coverage 100%)
- ✅ **epoch 10** (v3c con dang leo o epoch 4 → keo dai)
- ✅ **batch 20**, grad_accum 2 (effective 40)

⚠️ **LHP**: v3c VO TINH bat LHP (do bug config `false`→truthy, da fix). Giu **LHP=true** o day de KHONG tut
so voi v3c (LHP can bbox — manifest da co). Doi `LHP=false` o cell 5 neu muon tat that.

**VAL-B**: dung **dung split seed-42 nhu v3a/b/c** (621 query) → mAP so sanh truc tiep voi 0.6759/0.6763/0.7485.

**Add Input:** dataset `star-10k-hard` (tar.zst + xvlm_16m_base.th). **GPU T4 · Internet ON.**
**Thoi gian:** ~10' setup + ~3' extract + **train ~3.5–4.5h** (10 pair-epoch). Best.pth o `/kaggle/working/v6/best.pth`.

In [ ]:
# [1/7] GPU + clone/pull repo (can ban MOI: PairBatchSampler + FIX config bool override)
import os, pathlib, subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert gpu, "KHONG THAY GPU — bat Settings > Accelerator = GPU T4!"
print("GPU:", gpu)
os.chdir("/kaggle/working")
if pathlib.Path("star/.git").exists():
    !cd star && git pull -q
else:
    !git clone -q https://github.com/Khanhhh239/Model_XVLM_Training.git star
os.chdir("/kaggle/working/star")
sp = pathlib.Path("src/star/data/sampler.py").read_text()
cf = pathlib.Path("src/star/config.py").read_text()
assert "PairBatchSampler" in sp, "repo cu — thieu PairBatchSampler, pull lai!"
assert 'low == "true"' in cf, "repo cu — chua co FIX bool override, pull lai!"
print("repo OK (PairBatchSampler + bool-fix):", os.getcwd())

In [ ]:
# [2/7] Env pinned + X-VLM + 4 patches
!python scripts/kaggle_setup.py

In [ ]:
# [3/7] Tim dataset + giai nen tar.zst
import glob, os, pathlib, shutil
def find_one(p):
    h = sorted(set(glob.glob(f"/kaggle/input/*/{p}") + glob.glob(f"/kaggle/input/*/**/{p}", recursive=True)))
    return h[0] if h else None

CKPT  = find_one("xvlm_16m_base.th")
JSONL = find_one("train_10k_hard.jsonl")
ARCH  = find_one("train_10k_hard_data.tar.zst")
if JSONL:
    DATA_ROOT = str(pathlib.Path(JSONL).parents[2])
elif ARCH:
    marker = "/kaggle/working/extract"
    got = glob.glob(f"{marker}/**/train_10k_hard.jsonl", recursive=True)
    if not got:
        os.makedirs(marker, exist_ok=True)
        print("extracting 1.6GB (~2-4 phut)...")
        if shutil.which("zstd"):
            os.system(f"tar -I 'zstd -d' -xf '{ARCH}' -C {marker}")
        else:
            import zstandard, tarfile
            with open(ARCH, "rb") as fh, zstandard.ZstdDecompressor().stream_reader(fh) as zr:
                with tarfile.open(fileobj=zr, mode="r|") as tf:
                    tf.extractall(marker)
        got = glob.glob(f"{marker}/**/train_10k_hard.jsonl", recursive=True)
    assert got, "giai nen that bai"
    DATA_ROOT = str(pathlib.Path(got[0]).parents[2])
else:
    raise FileNotFoundError("Khong thay data! Add Input dataset star-10k-hard.")
if not CKPT:
    os.makedirs("data/checkpoints", exist_ok=True)
    !gdown -q 1iXgITaSbQ1oGPPvGaV0Hlae4QiJG5gx0 -O data/checkpoints/xvlm_16m_base.th
    CKPT = "data/checkpoints/xvlm_16m_base.th"
assert pathlib.Path(CKPT).stat().st_size > 8e8
print("DATA_ROOT =", DATA_ROOT, "| CKPT =", CKPT)

In [ ]:
# [4/7] Build manifest (GIONG HET v3: keypoints 100% + bbox + pair_image_id + split video seed-42)
import json, pathlib
import numpy as np, pandas as pd

root = pathlib.Path(DATA_ROOT)
anchors, hard_rows, anchor_ids = [], {}, set()
def bucket_of(p):
    return "wentwrong" if "/wentwrong/" in p else ("full" if "/full/" in p else "goal")

for line in open(root / "data/subsets/train_10k_hard.jsonl", encoding="utf-8"):
    r = json.loads(line)
    anchor_ids.add(r["image_id"])
    anchors.append(dict(image_path=r["image_webp"], caption=r["caption"],
        sequence_id=f'v{r["video_id"]}_{r["bucket"]}', scene=f'v{r["video_id"]}',
        action=str(r.get("label", "unk")), video_id=r["video_id"], image_id=r["image_id"],
        pair_image_id=r.get("hard_i_id")))
    hid = r.get("hard_i_id")
    if hid and hid not in hard_rows:
        hard_rows[hid] = dict(image_path=r["hard_image_webp"], caption=r["hard_c"],
            sequence_id=f'v{r["video_id"]}_{bucket_of(r["hard_image_webp"])}', scene=f'v{r["video_id"]}',
            action="hard_pair", video_id=r["video_id"], image_id=hid, pair_image_id=None)

df = pd.DataFrame(anchors + [v for k, v in hard_rows.items() if k not in anchor_ids])
pose = json.load(open(root / "data/subsets/prepared/train_10k_hard_vitpose.json", encoding="utf-8"))["items"]
def kpts_of(iid):
    it = pose.get(iid)
    if not it or it.get("status") != "ok" or not it.get("instances"):
        return None
    W, H = it.get("width", 384), it.get("height", 384)
    flat = []
    for x, y, c in it["instances"][0]["keypoints_xyc"]:
        flat += [x / W, y / H, c]
    return flat if len(flat) == 51 else None
def bbox_of(iid):
    it = pose.get(iid)
    b = it.get("primary_bbox_norm_xyxy") if it else None
    if not b:
        return None
    x1, y1, x2, y2 = b
    return [x1, y1, max(x2 - x1, 1e-3), max(y2 - y1, 1e-3)]
df["keypoints"] = df.image_id.map(kpts_of)
df["bbox"] = df.image_id.map(bbox_of)

rng = np.random.default_rng(42)
vids = df.video_id.unique().copy(); rng.shuffle(vids)
counts = df.groupby("video_id").size(); vq, vd, acc = set(), set(), 0
it = iter(vids)
for v in it:
    vq.add(v); acc += counts[v]
    if acc >= 600: break
acc = 0
for v in it:
    vd.add(v); acc += counts[v]
    if acc >= 900: break
df["split"] = np.where(df.video_id.isin(vq | vd), "valb", "train")
df.loc[df.video_id.isin(vd), "caption"] = ""

MANIFEST = "/kaggle/working/manifest_10k_hard.parquet"
df.to_parquet(MANIFEST, index=False)
leak = len(set(df[df.split == "train"].video_id) & set(df[df.split == "valb"].video_id))
n_pair = df[df.split == "train"].pair_image_id.notna().sum()
print(f"rows={len(df)} train={(df.split=='train').sum()} "
      f"valb_q={((df.split=='valb') & (df.caption!='')).sum()} valb_d={((df.split=='valb') & (df.caption=='')).sum()}")
print(f"kpts={df.keypoints.notna().mean():.0%} leakage={leak} anchor-co-pair(train)~{n_pair}")
assert leak == 0 and df.keypoints.notna().mean() > 0.99

In [ ]:
# [5/7] ===== CAU HINH V6 ===== (pair + text-LoRA + ViTPose + LHP, batch 20, epoch 10)
DATA = f"data.manifest={MANIFEST} data.image_root={DATA_ROOT} model.checkpoint={CKPT}"
LHP = "true"     # v3c vo tinh bat LHP; giu 'true' de khong tut. Doi 'false' de tat that (bug da fix).
V6 = (f"data.group_by=pair model.lora_freeze_text=false model.pose_enabled=true "
      f"data.lhp_enabled={LHP} loss.lambda_smooth_ap=0.2 optim.lr_lora=2e-4 "
      f"optim.epochs=10 train.early_stop_patience=10 train.eval_every_epochs=0.5 "
      f"train.batch_size=20 train.grad_accum=2 train.out_dir=/kaggle/working/v6")
print("V6:", V6)

In [ ]:
# [6/7] GATE overfit-one-batch (do VRAM dong 'vram='; text-LoRA them ~0.5-1G). OOM -> ha batch_size=16
gate = f"python scripts/train.py --config configs/star_v3_10k_kaggle.yaml --overfit-one-batch --set {DATA} {V6}"
print(gate)
!{gate}
print("Neu OOM: sua train.batch_size=16 trong V6 (cell 5) roi chay lai tu cell nay.")

In [ ]:
# [7/7] TRAIN V6 (~3.5-4.5h) -> best.pth. So sanh moc v3c=0.7485 / v3a=0.6759
import pathlib, time, torch
cmd = f"python scripts/train.py --config configs/star_v3_10k_kaggle.yaml --set {DATA} {V6}"
print(cmd); t0 = time.time()
!{cmd}
mins = round((time.time() - t0) / 60, 1)
BEST = "/kaggle/working/v6/best.pth"
assert pathlib.Path(BEST).exists(), "train fail — xem log (OOM? ha batch_size=16)"
raw = torch.load(BEST, map_location="cpu", weights_only=False)
rep = (raw.get("extra") or {}).get("report")
print("=" * 80)
print(f"V6 XONG ({mins}'): VAL-B best mAP = {raw.get('best_metric'):.4f}")
print(f"  report: {rep}")
print(f"  MOC: v3c=0.7485 (R@1 0.631) | v3a=0.6759 | v2a=0.6652")
print(f"  => {'CAI THIEN' if (raw.get('best_metric') or 0) > 0.7485 else 'CHUA vuot'} v3c")
print(f"\nTAI best.pth: Save Version -> Output -> /kaggle/working/v6/best.pth ({pathlib.Path(BEST).stat().st_size/1e6:.0f} MB)")
del raw
!ls -lh /kaggle/working/v6/best.pth

## Doc ket qua
- So **VAL-B mAP** voi **v3c=0.7485** (cung split seed-42 nen so truc tiep duoc).
- `> 0.7485` → combo (text-LoRA + epoch dai) co cong → giu lam baseline moi.
- `≈ 0.7485` → pair-batch da la tran tren 10K-hard synthetic; muon len nua phai sang **data that / sim2real**, khong phai them knob.
- Best.pth nay **train POSE-ON** → khi inference (V5) nho dua keypoints (YOLOv8-pose) vao.
- **Nhac**: day la VAL-B SYNTHETIC; van phai do tren old-test THAT (V5) de biet co transfer khong.